In [55]:
!pip install yfinance scikit-learn pandas numpy -q


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
import pickle
import warnings
warnings.filterwarnings('ignore')

# Paths 
BASE_DIR      = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

os.makedirs(RAW_DIR,       exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print(f"RAW_DIR       : {RAW_DIR}")
print(f"PROCESSED_DIR : {PROCESSED_DIR}")

RAW_DIR       : c:\Users\akbar\VSCode Project\RaksaDana\data\raw
PROCESSED_DIR : c:\Users\akbar\VSCode Project\RaksaDana\data\processed


In [57]:
tickers    = ["BBCA.JK", "BBRI.JK", "BMRI.JK"]
start_date = "2019-11-20"
end_date   = "2026-01-20"

In [58]:
raw = yf.download(tickers, start=start_date, end=end_date, progress=False)

price_data = {}
for ticker in tickers:
    df = raw.loc[:, (slice(None), ticker)].copy()
    df.columns = df.columns.droplevel(1)
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df.index = pd.to_datetime(df.index)
    price_data[ticker] = df

In [59]:
price_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume
Date,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000


In [60]:
# Save raw price data
for ticker in tickers:
    out_path = os.path.join(RAW_DIR, f'{ticker.replace(".", "_")}_raw.csv')
    price_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBCA_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BBRI_JK_raw.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\raw\BMRI_JK_raw.csv


In [61]:
fundamental = {}

for ticker in tickers:
    info = yf.Ticker(ticker).info
    fundamental[ticker] = {
        'ROE': info.get('returnOnEquity'),
        'EPS': info.get('trailingEps'),
        'DY' : info.get('dividendYield') / 100
    }

In [62]:
fundamental

{'BBCA.JK': {'ROE': 0.22972, 'EPS': 466.78, 'DY': 0.0546},
 'BBRI.JK': {'ROE': 0.18135999, 'EPS': 389.12, 'DY': 0.1306},
 'BMRI.JK': {'ROE': 0.21040002, 'EPS': 603.08, 'DY': 0.11220000000000001}}

In [63]:
merged_data = {}

for ticker in tickers:
    df = price_data[ticker].copy()
    df['ROE'] = fundamental[ticker]['ROE']
    df['EPS'] = fundamental[ticker]['EPS']
    df['DY']  = fundamental[ticker]['DY']
    merged_data[ticker] = df

In [64]:
merged_data['BBCA.JK'].tail()

Price,Open,High,Low,Close,Volume,ROE,EPS,DY
Date,,,,,,,,
2026-01-12,7736.332185,7760.283678,7664.477706,7688.429199,82935700,0.22972,466.78,0.0546
2026-01-13,7688.429532,7736.332520,7664.478038,7736.332520,80536300,0.22972,466.78,0.0546
2026-01-14,7688.429521,7712.381015,7616.575040,7664.478027,105724300,0.22972,466.78,0.0546
2026-01-15,7664.478038,7832.138495,7640.526544,7736.332520,151753100,0.22972,466.78,0.0546
2026-01-19,7736.332365,7784.235352,7640.526391,7784.235352,172100000,0.22972,466.78,0.0546


In [65]:
def add_features(df):
    df = df.copy()
    
    df['MA7']  = df['Close'].rolling(7).mean()
    df['MA20'] = df['Close'].rolling(20).mean()
    df['MA50'] = df['Close'].rolling(50).mean()

    delta = df['Close'].diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / (loss + 1e-9)))

    sma20          = df['Close'].rolling(20).mean()
    std20          = df['Close'].rolling(20).std()
    df['BB_upper'] = sma20 + 2 * std20
    df['BB_lower'] = sma20 - 2 * std20
    df['BB_width'] = df['BB_upper'] - df['BB_lower']

    df['Daily_Return'] = df['Close'].pct_change()
    df['Log_Return']   = np.log(df['Close'] / df['Close'].shift(1))
    df['Volume_MA7']   = df['Volume'].rolling(7).mean()

    ema12          = df['Close'].ewm(span=12, adjust=False).mean()
    ema26          = df['Close'].ewm(span=26, adjust=False).mean()
    df['MACD']         = ema12 - ema26
    df['MACD_signal']  = df['MACD'].ewm(span=9, adjust=False).mean()

    df.dropna(inplace=True)
    return df

featured_data = {}
for ticker in tickers:
    featured_data[ticker] = add_features(merged_data[ticker])

In [66]:
featured_data['BBCA.JK'].columns.tolist()

['Open',
 'High',
 'Low',
 'Close',
 'Volume',
 'ROE',
 'EPS',
 'DY',
 'MA7',
 'MA20',
 'MA50',
 'RSI',
 'BB_upper',
 'BB_lower',
 'BB_width',
 'Daily_Return',
 'Log_Return',
 'Volume_MA7',
 'MACD',
 'MACD_signal']

In [67]:
feature_cols = [
    'Open','High','Low','Close','Volume',
    'ROE','EPS','DY',
    'MA7','MA20','MA50',
    'RSI','BB_upper','BB_lower','BB_width',
    'Daily_Return','Log_Return','Volume_MA7',
    'MACD','MACD_signal'
]

scaled_data = {}
scalers     = {}

for ticker in tickers:
    df     = featured_data[ticker][feature_cols].copy()
    scaler = MinMaxScaler(feature_range=(0, 1))
    arr    = scaler.fit_transform(df)
    scaled_data[ticker] = pd.DataFrame(arr, columns=feature_cols, index=df.index)
    scalers[ticker]     = scaler

In [68]:
scaled_data['BBCA.JK'].tail()

,Open,High,Low,Close,Volume,ROE,EPS,DY,MA7,MA20,MA50,RSI,BB_upper,BB_lower,BB_width,Daily_Return,Log_Return,Volume_MA7,MACD,MACD_signal
Date,,,,,,,,,,,,,,,,,,,,
2026-01-12,0.640048,0.617691,0.646207,0.628886,0.109641,0.0,0.0,0.0,0.633507,0.639488,0.676196,0.464235,0.598708,0.687119,0.076479,0.282207,0.308306,0.194317,0.554653,0.508908
2026-01-13,0.632479,0.613710,0.646207,0.636380,0.106469,0.0,0.0,0.0,0.634659,0.639488,0.674506,0.406538,0.598708,0.687119,0.076479,0.353886,0.382985,0.200804,0.556791,0.511151
2026-01-14,0.632479,0.609729,0.638625,0.625139,0.139767,0.0,0.0,0.0,0.632931,0.639488,0.672815,0.433868,0.598708,0.687119,0.076479,0.293883,0.320566,0.218385,0.550850,0.511474
2026-01-15,0.628695,0.629633,0.642416,0.636380,0.200617,0.0,0.0,0.0,0.630625,0.640142,0.671394,0.400127,0.598521,0.688506,0.072193,0.366044,0.395516,0.247833,0.554970,0.512752
2026-01-19,0.640048,0.621672,0.642416,0.643874,0.227516,0.0,0.0,0.0,0.630049,0.638616,0.669706,0.536357,0.590724,0.692955,0.040011,0.353737,0.382831,0.290784,0.564205,0.516064


In [69]:
WINDOW_SIZE = 60
TARGET_COL  = 'Close'
TARGET_IDX  = feature_cols.index(TARGET_COL)

def create_sequences(scaled_df, window_size, target_idx):
    X, y = [], []
    arr  = scaled_df.values
    for i in range(window_size, len(arr)):
        X.append(arr[i - window_size : i, :])
        y.append(arr[i, target_idx])
    return np.array(X), np.array(y)

sequences = {}

for ticker in tickers:
    X, y  = create_sequences(scaled_data[ticker], WINDOW_SIZE, TARGET_IDX)
    split = int(len(X) * 0.8)
    sequences[ticker] = {
        'X_train': X[:split],  'y_train': y[:split],
        'X_test' : X[split:],  'y_test' : y[split:],
    }

In [70]:
for ticker in tickers:
    s = sequences[ticker]
    print(f"{ticker} → X_train: {s['X_train'].shape}, X_test: {s['X_test'].shape}")

BBCA.JK → X_train: (1100, 60, 20), X_test: (276, 60, 20)
BBRI.JK → X_train: (1100, 60, 20), X_test: (276, 60, 20)
BMRI.JK → X_train: (1100, 60, 20), X_test: (276, 60, 20)


In [71]:
output = {
    'sequences'    : sequences,
    'scalers'      : scalers,
    'feature_cols' : feature_cols,
    'target_col'   : TARGET_COL,
    'target_idx'   : TARGET_IDX,
    'window_size'  : WINDOW_SIZE,
    'featured_data': featured_data,
    'fundamental'  : fundamental,
    'tickers'      : tickers,
}

pkl_path = os.path.join(PROCESSED_DIR, 'preprocessed_data.pkl')
with open(pkl_path, 'wb') as f:
    pickle.dump(output, f)
print(f"Saved: {pkl_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\preprocessed_data.pkl


In [72]:
# Save featured CSVs
for ticker in tickers:
    out_path = os.path.join(PROCESSED_DIR, f'{ticker.replace(".", "_")}_featured.csv')
    featured_data[ticker].to_csv(out_path)
    print(f"Saved: {out_path}")

Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBCA_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BBRI_JK_featured.csv
Saved: c:\Users\akbar\VSCode Project\RaksaDana\data\processed\BMRI_JK_featured.csv
